###Enviroment setup

In [ ]:
# ==========================================
# 1. CONECTAR GOOGLE DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ==========================================
# 1. IMPORTS
# ==========================================
import os
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import random
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
from PIL import Image
import time
import sys
import torch.nn.functional as tfunc
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn.functional as func
# from tqdm import tqdm
from torch.nn.functional import kl_div, softmax, log_softmax
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Ejecutar siempre antes de empezar el entrenamiento
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
# ==========================================
# 3. DEFINIR RUTAS Y CONSTANTES
# ==========================================
CSV_PATH     = "/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/sample_labels.csv"
IMAGES_DIR   = "/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/images"
MODEL_PATH   = "/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/m-30012020-104001.pth"

Mounted at /content/drive
Using device: cuda
GPU: Tesla T4


###Fine Tuning

####Transformaciones ChexNet

In [ ]:
from PIL import Image, ImageEnhance, ImageOps
class XRaysPolicy(object):
    """ Randomly choose one of the best 24 Sub-policies on ImageNet.

        Example:
        >>> policy = XRaysPolicy()
        >>> transformed = policy(image)

        Example as a PyTorch Transform:
        >>> transform=transforms.Compose([
        >>>     transforms.Resize(256),
        >>>     XRaysPolicy(),
        >>>     transforms.ToTensor()])
    """
    def __init__(self, fillcolor=(128, 128, 128)):
        set_seed(42)
        self.policies = [
            SubPolicy(0.4, "posterize", 8, 0.6, "rotate", 9, fillcolor),
            SubPolicy(0.6, "equalize", 5, 0.6, "autocontrast", 5, fillcolor),
            SubPolicy(0.8, "equalize", 8, 0.6, "equalize", 3, fillcolor),

            SubPolicy(0.4, "equalize", 4, 0.8, "rotate", 8, fillcolor),
            # SubPolicy(0.6, "solarize", 3, 0.6, "equalize", 7, fillcolor),
            # SubPolicy(0.8, "posterize", 5, 1.0, "equalize", 2, fillcolor),
            # SubPolicy(0.2, "rotate", 3, 0.6, "solarize", 8, fillcolor),
            # SubPolicy(0.6, "equalize", 8, 0.4, "posterize", 6, fillcolor),

            SubPolicy(0.8, "rotate", 8, 0.4, "color", 0, fillcolor),
            SubPolicy(0.4, "rotate", 9, 0.6, "equalize", 2, fillcolor),
            SubPolicy(0.0, "equalize", 7, 0.8, "equalize", 8, fillcolor),
            # SubPolicy(0.6, "invert", 4, 1.0, "equalize", 8, fillcolor),
            SubPolicy(0.6, "color", 4, 1.0, "contrast", 8, fillcolor),

            SubPolicy(0.8, "rotate", 8, 1.0, "color", 2, fillcolor),
            # SubPolicy(0.8, "color", 8, 0.8, "solarize", 7, fillcolor),
            # SubPolicy(0.4, "sharpness", 7, 0.6, "invert", 8, fillcolor),
            SubPolicy(0.6, "shearX", 5, 1.0, "equalize", 9, fillcolor),
            SubPolicy(0.4, "color", 0, 0.6, "equalize", 3, fillcolor),

            # SubPolicy(0.4, "equalize", 7, 0.2, "solarize", 4, fillcolor),
            # SubPolicy(0.6, "solarize", 5, 0.6, "autocontrast", 5, fillcolor),
            # SubPolicy(0.6, "invert", 4, 1.0, "equalize", 8, fillcolor),
            SubPolicy(0.6, "color", 4, 1.0, "contrast", 8, fillcolor),
            SubPolicy(0.8, "equalize", 8, 0.6, "equalize", 3, fillcolor)
        ]


    def __call__(self, img):
        policy_idx = random.randint(0, len(self.policies) - 1)
        return self.policies[policy_idx](img)

    def __repr__(self):
        return "AutoAugment XRays Policy"
class SubPolicy(object):
    def __init__(self, p1, operation1, magnitude_idx1, p2, operation2, magnitude_idx2, fillcolor=(128, 128, 128)):
        ranges = {
            "shearX": np.linspace(0, 0.3, 10),
            "shearY": np.linspace(0, 0.3, 10),
            "translateX": np.linspace(0, 150 / 331, 10),
            "translateY": np.linspace(0, 150 / 331, 10),
            "rotate": np.linspace(0, 30, 10),
            "color": np.linspace(0.0, 0.9, 10),
            "posterize": np.round(np.linspace(8, 4, 10), 0).astype(int),
            "solarize": np.linspace(256, 0, 10),
            "contrast": np.linspace(0.0, 0.9, 10),
            "sharpness": np.linspace(0.0, 0.9, 10),
            "brightness": np.linspace(0.0, 0.9, 10),
            "autocontrast": [0] * 10,
            "equalize": [0] * 10,
            "invert": [0] * 10
        }

        # from https://stackoverflow.com/questions/5252170/specify-image-filling-color-when-rotating-in-python-with-pil-and-setting-expand
        def rotate_with_fill(img, magnitude):
            rot = img.convert("RGBA").rotate(magnitude)
            return Image.composite(rot, Image.new("RGBA", rot.size, (128,) * 4), rot).convert(img.mode)

        func = {
            "shearX": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, magnitude * random.choice([-1, 1]), 0, 0, 1, 0),
                Image.BICUBIC, fillcolor=fillcolor),
            "shearY": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, 0, magnitude * random.choice([-1, 1]), 1, 0),
                Image.BICUBIC, fillcolor=fillcolor),
            "translateX": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, magnitude * img.size[0] * random.choice([-1, 1]), 0, 1, 0),
                fillcolor=fillcolor),
            "translateY": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, 0, 0, 1, magnitude * img.size[1] * random.choice([-1, 1])),
                fillcolor=fillcolor),
            "rotate": lambda img, magnitude: rotate_with_fill(img, magnitude),
            "color": lambda img, magnitude: ImageEnhance.Color(img).enhance(1 + magnitude * random.choice([-1, 1])),
            "posterize": lambda img, magnitude: ImageOps.posterize(img, magnitude),
            "solarize": lambda img, magnitude: ImageOps.solarize(img, magnitude),
            "contrast": lambda img, magnitude: ImageEnhance.Contrast(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "sharpness": lambda img, magnitude: ImageEnhance.Sharpness(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "brightness": lambda img, magnitude: ImageEnhance.Brightness(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "autocontrast": lambda img, magnitude: ImageOps.autocontrast(img),
            "equalize": lambda img, magnitude: ImageOps.equalize(img),
            "invert": lambda img, magnitude: ImageOps.invert(img)
        }

        self.p1 = p1
        self.operation1 = func[operation1]
        self.magnitude1 = ranges[operation1][magnitude_idx1]
        self.p2 = p2
        self.operation2 = func[operation2]
        self.magnitude2 = ranges[operation2][magnitude_idx2]


    def __call__(self, img):
        if random.random() < self.p1: img = self.operation1(img, self.magnitude1)
        if random.random() < self.p2: img = self.operation2(img, self.magnitude2)
        return img



####Dataset

In [ ]:

from torch.utils.data import Dataset
from PIL import Image
import os

class Dataset(Dataset):
    def __init__(self, csv_file=None, dataframe=None, image_dir=None,
                 transform=None, transform_aug=None):
        """
        Dataset para clasificación binaria (1 salida).

        Args:
            csv_file (str): ruta al CSV con columnas ['Image Index', 'Finding Labels']
            dataframe (pd.DataFrame): DataFrame ya cargado
            image_dir (str): carpeta donde están las imágenes
            transform (callable): transformaciones base (ToTensor + normalize)
            transform_aug (callable): transformaciones de augmentación extra
        """
        if dataframe is not None:
            self.df = dataframe.copy()
        elif csv_file is not None:
            self.df = pd.read_csv(csv_file)
            self.df['label'] = self.df['Finding Labels'].apply(lambda x: 1 if 'Cardiomegaly' in x.split('|') else 0)
        else:
            raise ValueError("Debes proporcionar csv_file o dataframe")

        self.image_dir = image_dir
        self.transform = transform
        self.transform_aug = transform_aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['Image Index'])
        image = Image.open(img_path).convert('RGB')

        # Label binario como float para BCELoss
        label = torch.tensor(row['label'], dtype=torch.float32)

        # Imagen base transformada
        img1 = self.transform(image) if self.transform else image

        # Imagen augmentada adicional
        if self.transform_aug is not None:
            img2 = self.transform(self.transform_aug(image))
            return img1, img2, label

        return img1, label, row['Image Index']

#####Analisis de la proporción de los datos

In [ ]:

dataset = Dataset(csv_file=CSV_PATH, image_dir=IMAGES_DIR, transform=None)
counts = dataset.df['label'].value_counts()
print(counts)
print(f"\nTotal imágenes: {len(dataset)}")
print(f"Imágenes con Enfermedad: {counts.get(1, 0)}")
print(f"Imágenes sin Enfermedad: {counts.get(0, 0)}")
print(f"Porcentaje de Enfermedad: {counts[1] / len(dataset) * 100:.2f}%")



label
0    5465
1     141
Name: count, dtype: int64

Total imágenes: 5606
Imágenes con Enfermedad: 141
Imágenes sin Enfermedad: 5465
Porcentaje de Enfermedad: 2.52%


#####Partición para el modelo utilizando el 100% de los datos como etiquetados

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Cargar CSV original
df = pd.read_csv(CSV_PATH)

# Crear la columna binaria
df['label'] = df['Finding Labels'].apply(lambda x: 1 if 'Cardiomegaly' in x.split('|') else 0)

# --- Split train / val / test de forma estratificada ---
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,                  # 70% train, 30% resto
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,                  # 20% val, 10% test (del total)
    stratify=temp_df['label'],
    random_state=42
)

print(f"Train: {len(train_df)} muestras ({train_df['label'].mean()*100:.2f}% Cardiomegalia)")
print(f"Val:   {len(val_df)} muestras ({val_df['label'].mean()*100:.2f}% Cardiomegalia)")
print(f"Test:  {len(test_df)} muestras ({test_df['label'].mean()*100:.2f}% Cardiomegalia)")

Train: 3924 muestras (2.52% Cardiomegalia)
Val:   1121 muestras (2.50% Cardiomegalia)
Test:  561 muestras (2.50% Cardiomegalia)


In [ ]:
#ASIGNACION DE PESOSO PARA LA PÉRDIDA


# Contar ejemplos por clase
class_counts = df["label"].value_counts().sort_index().values
print("Class counts:", class_counts)

# Pesos inversamente proporcionales
class_weights = 1.0 / class_counts

# Normalizar (opcional pero recomendable)
class_weights = class_weights / class_weights.sum()

# Convertir a tensor
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)

Class counts: [5465  141]
Class weights: tensor([0.0252, 0.9748], device='cuda:0')


#####Particiones para Active Learning utilizando el 20,30,50 y 70% como datos ya etiquetados

In [ ]:


def create_al_partitions(csv_path, output_dir="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits", test_ratio=0.1, val_ratio=0.1, seed=42):
    """
    Genera splits para Active Learning con varios tamaños de labeled inicial:
    20%, 30%, 50%, 70%. Mantiene el mismo test set y guarda CSVs.

    Args:
        csv_path (str): ruta al CSV original
        output_dir (str): carpeta donde guardar los splits
        test_ratio (float): porcentaje de test set
        val_ratio (float): porcentaje de validation dentro del labeled inicial
        seed (int): semilla para reproducibilidad
    """
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # --- Cargar CSV ---
    df = pd.read_csv(csv_path)

    # Crear columna binaria: 1 = neumonía, 0 = No Finding
    df['label'] = df['Finding Labels'].apply(lambda x: 1 if 'Cardiomegaly' in x.split('|') else 0)

    # --- Split train/test fijo ---
    train_pool, test_df = train_test_split(
        df,
        test_size=test_ratio,
        stratify=df['label'],
        random_state=seed
    )

    test_df.to_csv(os.path.join(output_dir, "test_set.csv"), index=False)
    print(f"Test set guardado ({len(test_df)} muestras)")

    # --- Barajar el pool para generar subsets anidados ---
    rng = pd.Series(range(len(train_pool)))
    rng = rng.sample(frac=1, random_state=seed).values

    labelled_fracs = [0.2, 0.3, 0.5,0.6, 0.7]  # tamaños de labelled inicial

    for frac in labelled_fracs:
        n_labeled = int(len(train_pool) * frac)
        labeled_indices = rng[:n_labeled]
        unlabelled_indices = rng[n_labeled:]

        labelled_df = train_pool.iloc[labeled_indices]
        unlabelled_df = train_pool.iloc[unlabelled_indices]

        # Split labelled en train/val
        train_df, val_df = train_test_split(
            labelled_df,
            test_size=val_ratio,
            stratify=labelled_df['label'],
            random_state=seed
        )

        # Guardar CSVs
        labelled_df.to_csv(os.path.join(output_dir, f"labelled_{int(frac*100)}.csv"), index=False)
        unlabelled_df.to_csv(os.path.join(output_dir, f"unlabelled_{int(frac*100)}.csv"), index=False)
        train_df.to_csv(os.path.join(output_dir, f"train_{int(frac*100)}.csv"), index=False)
        val_df.to_csv(os.path.join(output_dir, f"val_{int(frac*100)}.csv"), index=False)

        print(f"\nFrac {int(frac*100)}% → Train: {len(train_df)}, Val: {len(val_df)}, "
              f"Labeled total: {len(labelled_df)}, Unlabelled: {len(unlabelled_df)}")


# ----------------------------
# Uso ejemplo
# ----------------------------
set_seed(42)
create_al_partitions(CSV_PATH, output_dir="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits", test_ratio=0.1, val_ratio=0.1, seed=42)

Test set guardado (561 muestras)

Frac 20% → Train: 908, Val: 101, Labeled total: 1009, Unlabelled: 4036

Frac 30% → Train: 1361, Val: 152, Labeled total: 1513, Unlabelled: 3532

Frac 50% → Train: 2269, Val: 253, Labeled total: 2522, Unlabelled: 2523

Frac 60% → Train: 2724, Val: 303, Labeled total: 3027, Unlabelled: 2018

Frac 70% → Train: 3177, Val: 354, Labeled total: 3531, Unlabelled: 1514


###Definir la arquitectura del modelo

In [ ]:
from torchvision.models import DenseNet121_Weights
class DenseNet121(nn.Module):

    def __init__(self, classCount, isTrained):

        super(DenseNet121, self).__init__()

        self.densenet121 = torchvision.models.densenet121(weights=DenseNet121_Weights.DEFAULT)

        kernelCount = self.densenet121.classifier.in_features

        self.densenet121.classifier = nn.Sequential(nn.Linear(kernelCount, classCount))

    def forward(self, x):
        x = self.densenet121(x)
        return x

###Entrenamiento del modelo

In [ ]:
# Define aquí tu carpeta de preferencia en Drive
OUTPUT_MODELS_DIR = "/content/drive/MyDrive/MachineLearning/ALmodels/"

# Crear la carpeta si no existe
if not os.path.exists(OUTPUT_MODELS_DIR):
    os.makedirs(OUTPUT_MODELS_DIR)

In [ ]:

def get_uda_loss(preds1, preds2, threshold=0.75):
    teacher_probs = func.softmax(preds1.detach(), dim=1)  # (N, 2)
    student_probs = func.softmax(preds2, dim=1)           # (N, 2)

    # Predicción más confiable del teacher
    max_probs, _ = teacher_probs.max(dim=1)  # probabilidad máxima por fila

    # Crear máscara de confianza
    mask = (max_probs > threshold)

    if mask.sum() == 0:
        return torch.tensor(0.0, device=preds2.device)

    # Cross-entropy entre teacher y student
    # Nota: teacher_probs se trata como soft target (prob distrib)
    return func.kl_div(
        student_probs[mask].log(),  # log para KL divergence
        teacher_probs[mask],
        reduction='batchmean'
    )
def linear_rampup(current_epoch, rampup_length=12):
    if rampup_length == 0:
        return 1.0
    else:
        return min(1.0, current_epoch / rampup_length)


class ChexnetTrainer ():

    #---- Train the densenet network
    #---- pathDirData - path to the directory that contains images
    #---- pathFileTrain - path to the file that contains image paths and label pairs (training set)
    #---- pathFileVal - path to the file that contains image path and label pairs (validation set)
    #---- nnArchitecture - model architecture 'DENSE-NET-121', 'DENSE-NET-169' or 'DENSE-NET-201'
    #---- nnIsTrained - if True, uses pre-trained version of the network (pre-trained on imagenet)
    #---- nnClassCount - number of output classes
    #---- trBatchSize - batch size
    #---- trMaxEpoch - number of epochs
    #---- transResize - size of the image to scale down to (not used in current implementation)
    #---- transCrop - size of the cropped image
    #---- launchTimestamp - date/time, used to assign unique name for the checkpoint file
    #---- checkpoint - if not None loads the model and continues training

    def train (traindf,valdf,traincsv,valcsv,image_dir, nnIsTrained, trBatchSize, trMaxEpoch, transResize, transCrop, porcentaje, checkpoint ,query_fn,nnClassCount=2,saiakera=1):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Crear modelo
        if saiakera==1:
          model = DenseNet121(14, nnIsTrained).to(device)
        else:
          model = DenseNet121(nnClassCount, nnIsTrained).to(device)

        # Si quieres cargar un checkpoint
        if checkpoint:
            chk = torch.load(checkpoint, map_location=device, weights_only=False)
            # Solo carga los pesos que coinciden, útil si cambias la última capa
            model.load_state_dict(chk['state_dict'], strict=False)
            # Congelar capas si quieres fine-tuning
        for param in model.parameters():
            param.requires_grad = False

        # Reemplazar la última capa de clasificación correctamente
        num_ftrs = model.densenet121.classifier[0].in_features
        model.densenet121.classifier = nn.Sequential(nn.Linear(num_ftrs, nnClassCount)).to(device)

        # Descongelar solo las capas que entrenarás
        for param in model.densenet121.classifier.parameters():
          param.requires_grad = True
        for param in model.densenet121.features.denseblock4.parameters():
          param.requires_grad = True
        for param in model.densenet121.features.norm5.parameters():
          param.requires_grad = True

        # Añadir Dropout en la última capa convolucional
        model.dropout_conv = nn.Dropout2d(p=0.3)

        # -------------------- Forward con dropout en denseblock4
        def forward_dropout(x):
          x = model.densenet121.features(x)
          x = model.dropout_conv(x)            # Dropout en la convolución final
          x = func.relu(x, inplace=True)
          x = func.adaptive_avg_pool2d(x, (1, 1))
          x = torch.flatten(x, 1)
          x = model.densenet121.classifier(x)
          return x

        # Reemplaza el forward original
        model.forward = forward_dropout

        #-------------------- SETTINGS: DATA TRANSFORMS

        normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        set_seed(42)
        transformList = []
        transformList.append(transforms.RandomResizedCrop(transCrop))
        transformList.append(transforms.RandomHorizontalFlip())
        transformList.append(transforms.ToTensor())
        transformList.append(normalize)
        transformSequencetr=transforms.Compose(transformList)

        transformList = []
        transformList.append(transforms.Resize(transResize))
        transformList.append(transforms.ToTensor())
        transformList.append(normalize)
        #transformList.append(transforms.TenCrop(transCrop))
        #transformList.append(transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops])))
        #transformList.append(transforms.Lambda(lambda crops: torch.stack([normalize(crop) for crop in crops])))
        transformSequence=transforms.Compose(transformList)

        transform_only_aug = transforms.Compose([XRaysPolicy()])
        transform_with_aug = transforms.Compose([
            XRaysPolicy(),
            transforms.RandomResizedCrop(transCrop),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize
        ])
        #-------------------- SETTINGS: DATASET BUILDERS

        datasetTrain =Dataset(dataframe=traindf,csv_file=traincsv, image_dir=IMAGES_DIR, transform=transform_with_aug)
        datasetTrainUnsup = Dataset(dataframe=traindf,csv_file=traincsv, image_dir=IMAGES_DIR, transform=transformSequencetr, transform_aug=transform_only_aug)
        datasetVal =   Dataset(dataframe=valdf, csv_file=valcsv, image_dir=IMAGES_DIR, transform=transformSequence)
        dataLoaderTrain = DataLoader(dataset=datasetTrain, batch_size=trBatchSize, shuffle=True, pin_memory=True,num_workers=4)
        dataLoaderUnsup = DataLoader(dataset=datasetTrainUnsup, batch_size=trBatchSize, shuffle=True,  pin_memory=True,num_workers=4)
        dataLoaderVal = DataLoader(dataset=datasetVal, batch_size=trBatchSize, shuffle=False, pin_memory=True,num_workers=8)

        #-------------------- SETTINGS: OPTIMIZER & SCHEDULER
        optimizer = optim.Adam ([
    {"params": model.densenet121.classifier.parameters(), "lr": 1e-3},
    {"params": model.densenet121.features.denseblock4.parameters(), "lr": 1e-4},
    {"params": model.densenet121.features.norm5.parameters(), "lr": 1e-4},
    ], betas=(0.9, 0.999), eps=1e-08, weight_decay=1e-5)
        # Cosine annealing
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=trMaxEpoch,    # número de epochs hasta el mínimo del coseno
        eta_min=5e-4         # learning rate mínimo
          )
        #         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=5, mode='min')
        #-------------------- SETTINGS: LOSS
        loss = nn.CrossEntropyLoss(weight=class_weights)


        #---- TRAIN THE NETWORK
        start_epoch = 0
        lossMIN = 100000
        max_auroc_mean = -1000
        for epochID in range (start_epoch, trMaxEpoch):
            set_seed(42 + epochID)


            ChexnetTrainer.epochTrain (model, dataLoaderTrain, optimizer, scheduler, trMaxEpoch, nnClassCount, loss, dataLoaderUnsup,epochID)
            lossVal, losstensor, aurocMean, best_threshold = ChexnetTrainer.epochVal (model, dataLoaderVal, optimizer, scheduler, trMaxEpoch, nnClassCount, loss)
            scheduler.step()
            #timestampTime = time.strftime("%H%M%S")
            #timestampDate = time.strftime("%d%m%Y")
            #timestampEND = timestampDate + '-' + timestampTime



            if lossVal < lossMIN:
                lossMIN = lossVal
                torch.save({
                 'epoch': epochID + 1,
                 'state_dict': model.state_dict(), },f'm-{porcentaje}_round{saiakera-1}_strategy{query_fn.__name__}.pth.tar')
                print ('Epoch [' + str(epochID + 1) + '] [save] [' + str(porcentaje) + '] loss= ' + str(lossVal))
            else:
                print ('Epoch [' + str(epochID + 1) + '] [----] [' + str(porcentaje) + '] loss= ' + str(lossVal))

    #--------------------------------------------------------------------------------

    def epochTrain (model, dataLoader, optimizer, scheduler, epochMax, classCount, loss, unsup_loader,epochID):
        unsup_ratio=0.7
        lambda_u = unsup_ratio * linear_rampup(epochID, rampup_length=7)
        model.train()
        iter_u = iter(unsup_loader)

        for batchID, (inputs, target,_) in enumerate (dataLoader):
            l_data_len = len(target)
            try:
                u_input_1, u_input_2, u_label = next(iter_u)
            except StopIteration:
                iter_u = iter(unsup_loader)
                u_input_1, u_input_2, u_label= next(iter_u)

            inputs = torch.cat([inputs, u_input_1, u_input_2])

            target = target.long().to(device)
            inputs = inputs.to(device)

            varInput = torch.autograd.Variable(inputs)
            varTarget = torch.autograd.Variable(target)
            varOutput = model(inputs)

            lossvalue = loss(varOutput[:l_data_len],varTarget.long())

            # -------------------------uda loss
            preds_unsup = varOutput[l_data_len:]
            preds1, preds2 = torch.chunk(preds_unsup, 2)
            loss_kl_div = get_uda_loss(preds1, preds2)
            lossvalue = lossvalue + (lambda_u*loss_kl_div)
            #--------------------------


            optimizer.zero_grad()
            lossvalue.backward()
            optimizer.step()

    #--------------------------------------------------------------------------------

    def epochVal (model, dataLoader, optimizer, scheduler, epochMax, classCount, loss):

        model.eval()
        lossVal = 0
        lossValNorm = 0
        losstensorMean = 0

        outGT, outPRED = [], []

        for i, (input_, target,_) in enumerate(dataLoader):
          with torch.no_grad():
            # asegurarte de que target es un tensor


            target = target.long().cuda()
            #bs, n_crops, c, h, w = input_.size()
            varInput = input_.cuda()

            out = model(varInput)
            #outMean = out.view(bs, n_crops, -1).mean(1)

            # Acumular los tensores en listas
            losstensor = loss(out, target)
            losstensorMean += losstensor.item()
            lossVal += losstensor.item()
            lossValNorm += 1


            outPRED.append(out.cpu())
            # Convertir target a one-hot para AUROC multiclase/binaria
            target_onehot = torch.zeros(target.size(0), classCount).cuda()
            target_onehot.scatter_(1, target.view(-1, 1), 1)
            outGT.append(target_onehot.cpu())

        outGT = torch.cat(outGT, 0)
        outPRED = torch.cat(outPRED, 0)


        outLoss = lossVal / lossValNorm
        losstensorMean = losstensorMean / lossValNorm
        if classCount == 1:
          probs = torch.sigmoid(outPRED)
        else:
          probs = torch.softmax(outPRED, dim=1)  # softmax para 2 clases o más

        # Calcular AUROC
        aurocIndividual, best_thresholds = ChexnetTrainer.computeAUROC(
        outGT.numpy(), probs.numpy(), classCount
    )
        aurocMean = np.mean(aurocIndividual)


        print ('AUROC mean ', aurocMean)

        return outLoss, losstensorMean, aurocMean,best_thresholds

    #--------------------------------------------------------------------------------

    #---- Computes area under ROC curve
    #---- dataGT - ground truth data
    #---- dataPRED - predicted data
    #---- classCount - number of classes

    def computeAUROC(dataGT, dataPRED, classCount=1):
      aurocs = []
      best_thresholds = []

      datanpGT = dataGT.cpu().numpy() if isinstance(dataGT, torch.Tensor) else np.array(dataGT)
      datanpPRED = dataPRED.cpu().numpy() if isinstance(dataPRED, torch.Tensor) else np.array(dataPRED)

      if classCount == 1:
          auroc = roc_auc_score(datanpGT, datanpPRED)
          fpr, tpr, thresholds = roc_curve(datanpGT, datanpPRED)
          J = tpr - fpr
          best_idx = J.argmax()
          best_threshold = thresholds[best_idx]

          aurocs.append(auroc)
          best_thresholds.append(best_threshold)
      else:
          for i in range(classCount):
              auroc = roc_auc_score(datanpGT[:, i], datanpPRED[:, i])
              fpr, tpr, thresholds = roc_curve(datanpGT[:, i], datanpPRED[:, i])
              J = tpr - fpr
              best_idx = J.argmax()
              best_threshold = thresholds[best_idx]

              aurocs.append(auroc)
              best_thresholds.append(best_threshold)

      return aurocs, best_thresholds


para utilizar este codigo, cambiar a esto previamente la linea para guardar el modelo:

f'm-{porcentaje}_round{saiakera-1}_strategy{query_fn}.pth.tar'

In [ ]:

traindf=train_df
valdf=val_df
image_dir=IMAGES_DIR

#---- Neural network parameters: type of the network, is it pre-trained
#---- on imagenet, number of classes
nnIsTrained = True

#---- Training settings: batch size, maximum number of epochs
trBatchSize = 16
trMaxEpoch = 10

#---- Parameters related to image transforms: size of the down-scaled image, cropped image
imgtransResize = 256
imgtransCrop = 224
ChexnetTrainer.train(traindf,valdf,None,None,image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 100, MODEL_PATH ,'Baseline')

KeyboardInterrupt: 

In [ ]:

checkpoint = f"m-{100}_round{0}_strategy{'Baseline'}.pth.tar"
model = DenseNet121(2, nnIsTrained).to(device)
modelCheckpoint = torch.load(checkpoint, weights_only=False)
model.load_state_dict(modelCheckpoint['state_dict'])

# Evaluate
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
transformList = []
transformList.append(transforms.Resize(imgtransResize))
transformList.append(transforms.ToTensor())
transformList.append(normalize)
transformSequence = transforms.Compose(transformList)

# Solución aplicada aquí:
datasetTest = Dataset(dataframe=test_df, image_dir=IMAGES_DIR, transform=transformSequence)
dataLoaderTest = DataLoader(dataset=datasetTest, batch_size=16, shuffle=False, pin_memory=True, num_workers=8, drop_last=False)

acc, rec, f1 = evaluate_model(model, dataLoaderTest)

In [ ]:
# Imprimir los resultados
print("="*30)
print("RESULTADOS DE LA EVALUACIÓN")
print("="*30)
print(f"Accuracy : {acc:.2f}%")
print(f"Recall   : {rec:.2f}%")
print(f"F1 Score : {f1:.4f}")
print("="*30)

RESULTADOS DE LA EVALUACIÓN
Accuracy : 92.69%
Recall   : 78.57%
F1 Score : 34.9206


###Active Learning

In [ ]:

@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
) -> float:
    """Return classification accuracy (0–100) on a DataLoader."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    correct = total = TP=FN=FP=0
    for X_batch, y_batch,_ in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds   = model(X_batch).argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total   += y_batch.size(0)
        TP += ((preds == 1) & (y_batch == 1)).sum().item()
        FP += ((preds == 1) & (y_batch == 0)).sum().item()
        FN += ((preds == 0) & (y_batch == 1)).sum().item()

    if total == 0:
        return 0.0, 0.0, 0.0  # evita división por cero

    acc = 100 * correct / total

    # Recall y Precision
    recall = 100 * TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = 100 * TP / (TP + FP) if (TP + FP) > 0 else 0.0

    # F1-score
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    return acc, recall, f1

In [ ]:
@torch.no_grad()
def get_probabilities(
    model: nn.Module,
    dataset: Dataset,
    batch_size: int =16,
) -> torch.Tensor:
    """
    Compute softmax class probabilities for every sample in `dataset`.

    Returns:
        probas : Tensor of shape (N, K) on CPU.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_probas = []
    for X_batch, _ ,_ in loader:
        logits = model(X_batch.to(device))
        all_probas.append(func.softmax(logits, dim=-1).cpu())

    return torch.cat(all_probas, dim=0)   # (N, K)

In [25]:
def plot_learning_curves(
    histories: dict[str, dict],
    title: str = 'Active Learning Curves',
) -> None:
    """Plot test accuracy vs. number of labeled samples for multiple strategies."""
    fig, ax = plt.subplots(figsize=(8, 4))
    markers = ['o', 's', '^', 'D', 'v']
    for (name, hist), marker in zip(histories.items(), markers):
        ax.plot(hist['n_labeled'], hist['accuracies'],
                marker=marker, label=name, linewidth=1.8)
    ax.set_xlabel('Number of labeled samples')
    ax.set_ylabel('Test acc (%)')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

####Querys

In [ ]:
def margin_query(
    model: nn.Module, unlabeled_set: Dataset, n: int
) -> np.ndarray:
    """
    Margin sampling: select the `n` samples where the gap between the
    top two predicted classes is smallest.

    Score: 1 - (p_(1) - p_(2))

    Args:
        model        : Trained model (outputs logits).
        unlabeled_set: Pool of unlabeled samples.
        n            : Number of samples to select.

    Returns:
        indices : np.ndarray of shape (n,) — local indices into `unlabeled_set`.
    """
    probas = get_probabilities(model, unlabeled_set)          # (N, K)
    top2=torch.topk(probas,k=2, dim=1)
    scores = 1.0 - (top2.values[:, 0] - top2.values[:, 1])                 # (N,)
    return scores.argsort(descending=True)[:n].numpy()
    raise NotImplementedError


def entropy_query(
    model: nn.Module, unlabeled_set: Dataset, n: int
) -> np.ndarray:
    """
    Entropy sampling: select the `n` samples with highest predictive entropy.

    Score: -sum_k p_k * log(p_k)

    Args:
        model        : Trained model (outputs logits).
        unlabeled_set: Pool of unlabeled samples.
        n            : Number of samples to select.

    Returns:
        indices : np.ndarray of shape (n,) — local indices into `unlabeled_set`.
    """
    eps=1e-8
    probas = get_probabilities(model, unlabeled_set)          # (N, K)
    scores = -torch.sum(probas * torch.log(probas + eps), dim=1)  # (N,)
    return scores.argsort(descending=True)[:n].numpy()

    raise NotImplementedError

In [ ]:


def mc_dropout_query_03(
    model: nn.Module,
    unlabeled_set,
    n: int,
    T: int = 20,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> np.ndarray:
    """
    MC Dropout (p=0.3) uncertainty query strategy.
    Aplica Dropout2d después de norm5 (última capa convolucional-normalizadora)
    solo durante inferencia, para estimar incertidumbre sin alterar BatchNorm.
    """

    # --- Activar dropout (solo temporalmente para inferencia) ---
    model.dropout_mc = nn.Dropout2d(p=0.3)

    # Guardar el forward original
    original_forward = model.forward

    # Definir nuevo forward con Dropout2d tras norm5
    def forward_mc_dropout(x):
        # Extraer features
        x = model.densenet121.features(x)
        # Aplicar dropout 2D después de norm5
        x = model.dropout_mc(x)
        # Continuar forward original
        relu = torch.nn.ReLU(inplace=True)
        x = relu(x)
        x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = model.densenet121.classifier(x)
        return x

    # Reemplazar temporalmente el forward
    model.forward = forward_mc_dropout
    model.train()  # activar dropout (aunque no entrenamos)

    # --- T stocastic forward passes ---
    all_probs = []
    for _ in range(T):
        probs = get_probabilities(model, unlabeled_set).to(device)  # (N, K)
        all_probs.append(probs)

    # --- Calcular media de predicciones ---
    probs_T = torch.stack(all_probs)  # (T, N, K)
    p_bar = probs_T.mean(dim=0)       # (N, K)

    # --- Calcular entropía de la media ---
    eps = 1e-10
    scores = -(p_bar * (p_bar + eps).log()).sum(dim=1)

    return scores.argsort(descending=True)[:n].cpu().numpy()
#@torch.no_grad()
# def mc_dropout_query_05(
#     model: nn.Module,
#     unlabeled_set,
#     n: int,
#     T: int = 20,
#     device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
# ) -> np.ndarray:
#     """
#     MC Dropout (p=0.5) uncertainty query strategy.
#     Aplica Dropout2d después de norm5 (última capa convolucional-normalizadora)
#     solo durante inferencia, para estimar incertidumbre sin alterar BatchNorm.
#     """

#     # --- Activar dropout (solo temporalmente para inferencia) ---
#     model.dropout_mc = nn.Dropout2d(p=0.5)

#     # Guardar el forward original
#     original_forward = model.forward

#     # Definir nuevo forward con Dropout2d tras norm5
#     def forward_mc_dropout(x):
#         # Extraer features
#         x = model.densenet121.features(x)
#         # Aplicar dropout 2D después de norm5
#         x = model.dropout_mc(x)
#         # Continuar forward original
#         relu = torch.nn.ReLU(inplace=True)
#         x = relu(x)
#         x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
#         x = torch.flatten(x, 1)
#         x = model.densenet121.classifier(x)
#         return x

#     # Reemplazar temporalmente el forward
#     model.forward = forward_mc_dropout
#     model.train()  # activar dropout (aunque no entrenamos)

#     # --- T stocastic forward passes ---
#     all_probs = []
#     for _ in range(T):
#         probs = get_probabilities(model, unlabeled_set).to(device)  # (N, K)
#         all_probs.append(probs)

#     # --- Calcular media de predicciones ---
#     probs_T = torch.stack(all_probs)  # (T, N, K)
#     p_bar = probs_T.mean(dim=0)       # (N, K)

#     # --- Calcular entropía de la media ---
#     eps = 1e-10
#     scores = -(p_bar * (p_bar + eps).log()).sum(dim=1)

#     return scores.argsort(descending=True)[:n].cpu().numpy()


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics.pairwise import euclidean_distances


@torch.no_grad()
def medal_query(
    model: torch.nn.Module,
    unlabeled_set,
    n: int,
    labeled_features: np.ndarray,
    M: int = 500,               # número de muestras más inciertas a considerar
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> np.ndarray:
    """
    MedAL query strategy (entropía + diversidad).

    Usa entropy_query() para seleccionar las M muestras más inciertas.
    Calcula s(x) = (1/N) Σ_i d(f(x), f(x_i^train)) en ese subconjunto.
    Devuelve las n muestras con mayor s(x).

    Args:
        model            : Modelo entrenado (DenseNet121, etc.)
        unlabeled_set    : Dataset sin etiquetas.
        n                : Número de muestras a seleccionar.
        labeled_features : np.ndarray (N_labeled, D) con embeddings del conjunto etiquetado.
        M                : Número de muestras con mayor entropía a considerar.
        batch_size       : Tamaño del batch.
        device           : CPU o GPU.

    Returns:
        np.ndarray con índices globales de las 'n' muestras seleccionadas.
    """

    # Usar entropy_query para obtener las M más inciertas
    top_M_indices = entropy_query(model, unlabeled_set, M)
    subset = Subset(unlabeled_set, top_M_indices)

    # Extraer embeddings f(x) de esas M muestras (tras denseblock3)
    model.eval()
    model.to(device)
    loader = DataLoader(subset, batch_size=16, shuffle=False)
    feats_subset = []

    for imgs, _ ,_ in loader:
        imgs = imgs.to(device)
        with torch.no_grad():
            x = model.densenet121.features.conv0(imgs)
            x = model.densenet121.features.norm0(x)
            x = model.densenet121.features.relu0(x)
            x = model.densenet121.features.pool0(x)
            x = model.densenet121.features.denseblock1(x)
            x = model.densenet121.features.transition1(x)
            x = model.densenet121.features.denseblock2(x)
            x = model.densenet121.features.transition2(x)
            x = model.densenet121.features.denseblock3(x)
            pooled = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
        feats_subset.append(pooled.cpu().numpy())

    feats_subset = np.concatenate(feats_subset, axis=0)  # (M, D)

    # Calcular s(x) = (1/N) * Σ dist_euclídea entre f(x) y f(labelled)
    dist_matrix = euclidean_distances(feats_subset, labeled_features)  # (M, N_lab)
    s_values = dist_matrix.mean(axis=1)    # (M,)

    # Obtener los índices de las n muestras más dispares (mayor s)
    selected_subset_indices = np.argsort(s_values)[-n:]
    selected_global_indices = np.array(top_M_indices)[selected_subset_indices]

    return selected_global_indices

In [ ]:
# def least_confidence_query(
#     model: nn.Module, unlabeled_set: Dataset, n: int
# ) -> np.ndarray:
#     """
#     Least-confidence sampling: select the `n` samples for which the model's
#     top-class probability is lowest.

#     Score: 1 - max_k p_k
#     """

#     probas = get_probabilities(model, unlabeled_set)          # (N, K)
#     scores = 1.0 - probas.max(dim=1).values               # (N,)
#     return scores.argsort(descending=True)[:n].numpy()

def random_query(
     model: nn.Module, unlabeled_set: Dataset, n: int
 ) -> np.ndarray:
     """Random baseline: select `n` samples uniformly at random."""
     return np.random.choice(len(unlabeled_set), n, replace=False)

####Aplicar Active Learning

In [ ]:

def run_active_learning(
    query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    traindf, valdf,testdf,traincsv,valcsv,testcsv,unlabeledcsv, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, porcentaje, checkpoint,
    n_rounds: int = 10,
    query_size: int = 50,
    train_epochs: int = 10,
    train_batch_size: int = 16,

    verbose: bool = True,
) -> dict:

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    history = {'n_labeled': [], 'accuracies': [], 'recalls':[], 'F1':[]}


    for round_idx in range(n_rounds + 1):  # +1: evaluate initial labeled pool
        # Train on the current labeled pool
        ChexnetTrainer.train(traindf, valdf,traincsv,valcsv, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, porcentaje, checkpoint,query_fn=query_fn, saiakera=round_idx+1)
        checkpoint = f'm-{porcentaje}_round{round_idx}_strategy{query_fn.__name__}.pth.tar'
        model = DenseNet121(2, nnIsTrained).to(device)
        modelCheckpoint = torch.load(checkpoint,weights_only=False)
        model.load_state_dict(modelCheckpoint['state_dict'])


        # Evaluate
        normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        transformList = []
        transformList.append(transforms.Resize(imgtransResize))
        transformList.append(transforms.ToTensor())
        transformList.append(normalize)
        transformSequence=transforms.Compose(transformList)

        datasetTest =   Dataset(dataframe=testdf,csv_file=testcsv, image_dir=IMAGES_DIR, transform=transformSequence)
        dataLoaderTest = DataLoader(dataset=datasetTest, batch_size=trBatchSize, shuffle=False, pin_memory=True,num_workers=8,drop_last=False)
        acc,rec,f1 = evaluate_model(model, dataLoaderTest)

         # --- Save history ---
        history['n_labeled'].append(len(traindf))  # tamaño del dataset de entrenamiento actual
        history['accuracies'].append(acc)
        history['recalls'].append(rec)
        history['F1'].append(f1)
        bestf1=-1
        if f1 > bestf1:
          best_f1 = f1
          best_model_path = os.path.join(OUTPUT_MODELS_DIR, f'best_model{porcentaje}_strategy_{query_fn.__name__}_F1_{f1:.4f}.pth')
          torch.save(model.state_dict(), best_model_path)
        if verbose:
            print(f"Round {round_idx:2d} | Labeled: {len(traindf):4d} | Test acc: {acc:.2f}% |Test rec:{rec:.2f} | Test F1:{f1:.2f}")

        # --- Stop after last round ---
        if round_idx == n_rounds:
            break
        unlabeled_df=Dataset(dataframe=None,csv_file=unlabeledcsv, image_dir=IMAGES_DIR, transform=transformSequence)
        # Query
        actual_query = min(query_size, len(unlabeled_df))  # reemplaza 'unlabeled_df' por tu DataFrame sin etiquetar
        print(f"Querying {actual_query} samples...")
        if query_fn.__name__ == "medal_query":
            model.eval()
            loader_train = DataLoader(
                Dataset(dataframe=traindf, csv_file=traincsv,
                        image_dir=IMAGES_DIR, transform=transformSequence),
                batch_size=trBatchSize, shuffle=False, num_workers=8
            )
            feats_train = []
            with torch.no_grad():
                for imgs, _,_ in loader_train:
                    imgs = imgs.to(device)
                    # Features tras denseblock3
                    x = model.densenet121.features.conv0(imgs)
                    x = model.densenet121.features.norm0(x)
                    x = model.densenet121.features.relu0(x)
                    x = model.densenet121.features.pool0(x)
                    x = model.densenet121.features.denseblock1(x)
                    x = model.densenet121.features.transition1(x)
                    x = model.densenet121.features.denseblock2(x)
                    x = model.densenet121.features.transition2(x)
                    x = model.densenet121.features.denseblock3(x)
                    pooled = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
                    feats_train.append(pooled.cpu().numpy())

            labeled_features = np.concatenate(feats_train, axis=0)
            print(f"Extracted train features shape: {labeled_features.shape}")

            # Aquí se llama a MedAL con labeled_features
            selected_indices = query_fn(
                model, unlabeled_df, actual_query, labeled_features=labeled_features
            )

        # ------------------------------------------------------------------
        # 🔹 Para el resto de estrategias
        # ------------------------------------------------------------------
        else:
            selected_indices = query_fn(model, unlabeled_df, actual_query)

        # --- Actualizar el conjunto de entrenamiento ---
        new_train_samples = unlabeled_df.df.iloc[selected_indices]
        traindf = pd.concat([traindf, new_train_samples], ignore_index=True)
        unlabeled_df = unlabeled_df.df.drop(selected_indices).reset_index(drop=True)

    return history

###20% Para entrenamiento

In [ ]:
train20="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/train_20.csv"
val20="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/val_20.csv"
test="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/test_set.csv"
unlabeled20="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/unlabelled_20.csv"

# Convertir a DataFrames
train20_df = pd.read_csv(train20)
val20_df = pd.read_csv(val20)
test_df = pd.read_csv(test)
unlabeled20_df = pd.read_csv(unlabeled20)

In [ ]:

image_dir=IMAGES_DIR

#---- Neural network parameters: type of the network, is it pre-trained
#---- on imagenet, number of classes
nnIsTrained = True

#---- Training settings: batch size, maximum number of epochs
trBatchSize = 16
trMaxEpoch = 10

#---- Parameters related to image transforms: size of the down-scaled image, cropped image
imgtransResize = 256
imgtransCrop = 224
checkpoint = MODEL_PATH
#ChexnetTrainer.train(traindf, valdf,train20,val20, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 20, checkpoint)

In [ ]:
import shutil
STRATEGIES = {
    'Random':           random_query,
    'Margin':           margin_query,
    'Entropy':          entropy_query,
    'MC Dropout 0.3':  mc_dropout_query_03,
    'MedAL':            medal_query,
}


histories_20 = {}

for name, query_fn in STRATEGIES.items():
    print(name)
    checkpoint_copy = f"{name}_init.pth.tar"
    shutil.copyfile(MODEL_PATH, checkpoint_copy)
    set_seed(42)
    hist = run_active_learning(query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    train20_df.copy(),
    val20_df.copy(),
    test_df.copy(),None,None,None,unlabeled20, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 20, checkpoint,
    n_rounds= 5,
    query_size= 100,
    verbose= True,
    )
    histories_20[name] = hist

plot_learning_curves(histories_20, title='AL con 20% de labeled data')

Este bloque de aqui lo utilizamos para poder solventar el error que nos ha saltado antes y no perder el historial

In [ ]:
STRATEGIES = {
    'MedAL':            medal_query,
}


for name, query_fn in STRATEGIES.items():
    print(name)
    checkpoint_copy = f"{name}_init.pth.tar"
    shutil.copyfile(MODEL_PATH, checkpoint_copy)
    set_seed(42)
    hist = run_active_learning(query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    train20_df.copy(),
    val20_df.copy(),
    test_df.copy(),None,None,None,unlabeled20, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 20, checkpoint,
    n_rounds= 5,
    query_size= 100,
    verbose= True,
    )
    histories_20[name] = hist

plot_learning_curves(histories_20, title='AL con 20% de labeled data')

In [ ]:
plot_learning_curves(histories_20, title='AL con 20% de labeled data')

###30% Para entrenamiento

In [ ]:
train30="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/train_30.csv"
val30="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/val_30.csv"
unlabeled30="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/unlabelled_30.csv"

# Convertir a DataFrames
train30_df = pd.read_csv(train30)
val30_df = pd.read_csv(val30)
unlabeled30_df = pd.read_csv(unlabeled30)

In [ ]:

STRATEGIES = {
    'Random':           random_query,
    'Margin':           margin_query,
    'Entropy':          entropy_query,
    'MC Dropout 0.3':  mc_dropout_query_03,
    'MedAL':            medal_query,
}


histories_30 = {}

for name, query_fn in STRATEGIES.items():
    print(name)
    checkpoint_copy = f"{name}_init.pth.tar"
    shutil.copyfile(MODEL_PATH, checkpoint_copy)
    set_seed(42)
    hist = run_active_learning(query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    train30_df.copy(),
    val30_df.copy(),
    test_df.copy(),None,None,None,unlabeled30, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 30, checkpoint,
    n_rounds= 5,
    query_size= 100,
    verbose= True,
    )
    histories_30[name] = hist

plot_learning_curves(histories_30, title='AL con 30% de labeled data')

###50% Para entrenamiento

In [ ]:
train50="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/train_50.csv"
val50="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/val_50.csv"
unlabeled50="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/unlabelled_50.csv"

# Convertir a DataFrames
train50_df = pd.read_csv(train50)
val50_df = pd.read_csv(val50)
unlabeled50_df = pd.read_csv(unlabeled50)

In [ ]:

STRATEGIES = {
    'Random':           random_query,
    'Margin':           margin_query,
    'Entropy':          entropy_query,
    'MC Dropout 0.3':  mc_dropout_query_03,
    'MedAL':            medal_query,
}


histories_50 = {}

for name, query_fn in STRATEGIES.items():
    print(name)
    checkpoint_copy = f"{name}_init.pth.tar"
    shutil.copyfile(MODEL_PATH, checkpoint_copy)
    set_seed(42)
    hist = run_active_learning(query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    train50_df.copy(),
    val50_df.copy(),
    test_df.copy(),None,None,None,unlabeled50, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 50, checkpoint,
    n_rounds= 5,
    query_size= 100,
    verbose= True,
    )
    histories_50[name] = hist

plot_learning_curves(histories_50, title='AL con 50% de labeled data')

###60% Para entrenamiento

In [ ]:
train60="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/train_60.csv"
val60="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/val_60.csv"
unlabeled60="/content/drive/MyDrive/MachineLearning/Part3/CHEXNET/archive (Unzipped Files)/splits/unlabelled_60.csv"

# Convertir a DataFrames
train60_df = pd.read_csv(train60)
val60_df = pd.read_csv(val60)
unlabeled60_df = pd.read_csv(unlabeled60)

In [ ]:

STRATEGIES = {
    'Random':           random_query,
    'Margin':           margin_query,
    'Entropy':          entropy_query,
    'MC Dropout 0.3':  mc_dropout_query_03,
    'MedAL':            medal_query,
}


histories_60 = {}

for name, query_fn in STRATEGIES.items():
    print(name)
    checkpoint_copy = f"{name}_init.pth.tar"
    shutil.copyfile(MODEL_PATH, checkpoint_copy)
    set_seed(42)
    hist = run_active_learning(query_fn,               # callable (model, unlabeled_set, n) → np.ndarray of local indices
    train60_df.copy(),
    val60_df.copy(),
    test_df.copy(),None,None,None,unlabeled60, image_dir, nnIsTrained, trBatchSize, trMaxEpoch, imgtransResize, imgtransCrop, 60, checkpoint,
    n_rounds= 5,
    query_size= 100,
    verbose= True,
    )
    histories_60[name] = hist

plot_learning_curves(histories_60, title='AL con 60% de labeled data')